# Finite-temperature phonons with pheasy and machine-learned potentials

This notebook computes phonons at a finite temperature with the
`ForceFieldFiniteTemperaturePhononMaker` workflow. The forces come from three
MACE potentials, so no DFT is needed.

Section 1 runs five materials at 100, 300, 500, 700 and 900 K. Ca3Ir4Sn13 is
worked through step by step. Section 2 tests the length of the MD and the number
of snapshots for KNaICl.

This workflow is new. It is less tested than the other phonon workflows in
atomate2, and its defaults and output may still change.

## Background

Harmonic phonons from finite displacements describe the crystal at 0 K. Many
materials have imaginary modes at 0 K. The anharmonic motion of the atoms can
stabilize these modes at higher temperature. This workflow fits effective
harmonic force constants to an MD trajectory at the temperature of interest, as
in the temperature-dependent effective potential (TDEP) method of Hellman et
al., Phys. Rev. B 84, 180301 (2011). These force constants include the
anharmonic effects at that temperature.

The flow has these steps:

1. The structure is relaxed. The relaxation can be switched off, as done below.
2. A diagonal supercell is built. Its lattice vectors are at least `min_length`
   long, 12 Å by default. A supercell matrix can also be given.
3. An NVT MD run of the supercell at the temperature. The defaults are 8 ps with
   a 1 fs time step and a Nose-Hoover thermostat.
4. The first 1 ps of the MD is left out. 50 snapshots are picked, evenly spread
   over the rest of the trajectory.
5. The forces on the snapshots and on the undisplaced supercell are computed.
6. pheasy fits the second-order force constants with LASSO. The displacements
   are measured from the undisplaced supercell. The forces of the undisplaced
   supercell are subtracted from the forces of each snapshot.
7. The band structure and the density of states (DOS) follow from the force
   constants. Imaginary modes are reported. They are not removed.

The MD trajectory is also checked. The check looks for melting and for a move
away from the starting structure. It also looks at the drift of the potential
energy late in the run. The result is stored in the output. A warning is raised
if the check fails, but the fit is still done.

The volume stays that of the starting structure, so thermal expansion is not
included. The same workflow runs with VASP as
`atomate2.vasp.flows.finite_temperature_phonons.FiniteTemperaturePhononMaker`.
`atomate2.forcefields.flows.finite_temperature_phonons` also has two mixed
versions. One runs the MD with VASP and the force calculations with a force
field. The other one does the reverse.

## Installation

The `pheasy` and `ase` extras are needed, plus MACE and the Materials Project
client:

```
pip install 'atomate2[pheasy,ase]'
pip install 'mace-torch>=0.3.16' mp-api
```

The `pheasy` extra pulls in pheasy and ALM. ALM is compiled from source. The
hiPhive tutorial and the atomate2 VASP documentation list what the build needs.

## The potentials

The notebook uses three MACE foundation models. All three are released under the
Academic Software License.

- **MACE-OMAT-0-medium.** `mace_mp(model="medium-omat-0")` downloads it once and
  caches it under `~/.cache/mace`.
- **MACE-MATPES-PBE-0** and **MACE-MATPES-r2SCAN-0.** These are MACE-OMAT-0
  fine-tuned on the MatPES data set at the PBE and r2SCAN levels. They need
  `mace-torch>=0.3.10`. Download the two files from the `mace_matpes_0` release of
  [mace-foundations](https://github.com/ACEsuit/mace-foundations/releases/tag/mace_matpes_0):
  - [MACE-matpes-pbe-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-pbe-omat-ft.model)
  - [MACE-matpes-r2scan-omat-ft.model](https://github.com/ACEsuit/mace-foundations/releases/download/mace_matpes_0/MACE-matpes-r2scan-omat-ft.model)

Set the paths to the two downloaded files in the next cell. The Ca3Ir4Sn13
example below only needs MACE-OMAT-0-medium.

The MD takes most of the time, so a GPU helps a lot. The next cell uses one if
PyTorch finds it. All our runs used `float64` on one NVIDIA A100 GPU of a
Perlmutter node.

In [ ]:
import os
import warnings

# macOS: conda's llvm-openmp and torch's bundled libomp both load and the
# duplicate aborts the process. This must be set before torch is imported.
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
warnings.filterwarnings("ignore")

import torch  # noqa: E402
from mace.calculators import mace_mp  # noqa: E402

# Replace the two paths with the files you downloaded.
MODEL_FILES = {
    "OMAT-0-medium": "medium-omat-0",
    "MATPES-PBE-0": "/path/to/MACE-matpes-pbe-omat-ft.model",
    "MATPES-r2SCAN-0": "/path/to/MACE-matpes-r2scan-omat-ft.model",
}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def calc_kwargs(model: str) -> dict:
    """Return the calculator settings for one of the models above."""
    return {"model": MODEL_FILES[model], "device": DEVICE, "default_dtype": "float64"}


_ = mace_mp(**calc_kwargs("OMAT-0-medium"))  # downloads and caches on first use

## One calculator for all jobs

Every force field job builds its own calculator. `run_locally` runs all of
them in this one Python process, and the workflow has a little over 50 jobs.
Loading MACE for each of them grows the memory by about 150 MB per job. This
cell makes the jobs share one calculator. It is only needed when many jobs run
in one process, not with a workflow manager.

In [ ]:
from ase.calculators.calculator import Calculator

import atomate2.forcefields.utils as ff_utils

_ase_calculator = ff_utils.ase_calculator
_calculators: dict[str, Calculator] = {}


def shared_ase_calculator(calculator_meta: object, **kwargs: object) -> Calculator:
    """Build each calculator once and reuse it for every job."""
    key = repr((calculator_meta, sorted(kwargs.items())))
    if key not in _calculators:
        _calculators[key] = _ase_calculator(calculator_meta, **kwargs)
    _calculators[key].reset()
    return _calculators[key]


ff_utils.ase_calculator = shared_ase_calculator

## 1. Five materials from 100 K to 900 K

The first part runs Ca3Ir4Sn13 at 300 K with MACE-OMAT-0-medium step by step.
The other temperatures, potentials and materials follow with the same code.

## The structure

Ca3Ir4Sn13 (mp-1200211) is cubic, with 40 atoms in the primitive cell. We start
from its PBEsol cell in the Materials Project's Harmonic Phonon Database. The
phonon documents of the Materials Project hold this cell. Set your key first with
`export MP_API_KEY=...`.

In [ ]:
from mp_api.client import MPRester
from pymatgen.core import Structure

if not os.environ.get("MP_API_KEY"):
    raise OSError("export MP_API_KEY before running this cell")


def get_pheasy_structure(mp_id: str) -> Structure:
    """Get the PBEsol cell of a material in the Harmonic Phonon Database."""
    with MPRester() as mpr:
        (doc,) = mpr.materials.phonon.search(
            material_ids=[mp_id],
            phonon_method="pheasy",
            fields=["structure"],
            num_chunks=1,
            chunk_size=10,
        )
    return doc.structure


structure = get_pheasy_structure("mp-1200211")
structure.lattice

## Building the workflow

`ForceFieldFiniteTemperaturePhononMaker.from_force_field_name` sets one force
field for the relaxation, the MD and the force calculations. Any MACE model
routes through `mace_mp`, and the model name in `calculator_kwargs` selects the
weights.

We skip the relaxation with `relax_initial_structure=False`. The cell is
already relaxed with PBEsol, and our AIMD reference below uses the same cell. If
your input cell is not properly relaxed, keep the default
`relax_initial_structure=True`. The force field then relaxes the cell before
the MD.

We give the supercell matrix of our AIMD reference, 2x2x2 with 320 atoms.
Everything else stays at the workflow defaults.

In [ ]:
from atomate2.forcefields.flows.finite_temperature_phonons import (
    ForceFieldFiniteTemperaturePhononMaker,
)

maker = ForceFieldFiniteTemperaturePhononMaker.from_force_field_name(
    "MACE-MP-0",
    calculator_kwargs=calc_kwargs("OMAT-0-medium"),
    relax_initial_structure=False,
    temperature=300,
)
flow = maker.make(structure, supercell_matrix=[[2, 0, 0], [0, 2, 0], [0, 0, 2]])
flow.draw_graph().show()

## Running the workflow

The flow has one MD job and 51 force calculations, 50 snapshots and the
undisplaced supercell. On one A100 GPU the whole run took 11 minutes. The MD
took 9 minutes, the force calculations 11 seconds and the fit 2 minutes.

`create_folders=True` is needed, because the snapshots are read from the
trajectory file in the folder of the MD job.

In [ ]:
from jobflow import JobStore, run_locally
from maggma.stores import MemoryStore

job_store = JobStore(MemoryStore(), additional_stores={"data": MemoryStore()})
responses = run_locally(flow, store=job_store, create_folders=True, ensure_success=True)
doc = responses[flow.output.uuid][1].output

## Results

The output is a `FiniteTemperaturePhononDoc`. It holds the band structure, the
DOS, the force constants, the fit and the trajectory check.

In [ ]:
import numpy as np

health = doc.trajectory_health
{
    "highest frequency in THz": float(np.max(doc.phonon_bandstructure.bands)),
    "imaginary modes on the band path": doc.has_imaginary_modes,
    "lowest frequency at the commensurate q-points in THz": doc.min_frequency,
    "LASSO penalty": doc.lasso_alpha,
    "force RMSE in eV/A": doc.force_rmse,
    "trajectory check": health.verdict,
    "Lindemann ratio": health.lindemann_ratio,
}

Our run gave a highest frequency of 5.97 THz, no imaginary modes and a stable
trajectory.

- `has_imaginary_modes` checks the whole band path. `min_frequency` only covers
  the q-points commensurate with the supercell. The fit sets the frequencies at
  these points directly. Everywhere else they are interpolated. Section 2 shows
  why the difference matters.
- The force RMSE is the part of the forces that the force constants do not
  describe. It grows with temperature, from 0.05 eV/Å at 100 K to 0.29 eV/Å at
  900 K for Ca3Ir4Sn13.
- The Lindemann ratio is the root mean square vibration of the atoms around
  their mean positions, divided by the nearest-neighbor distance. Above 0.12 the check reports the trajectory as
  melted.

## Comparing with 0 K

The Materials Project's Harmonic Phonon Database has the 0 K phonons of the same
cell, computed with PBEsol (Sahasrabuddhe et al., ChemRxiv (2026),
doi:10.26434/chemrxiv.15004632/v1). The next cell plots them next to ours. The
two band structures use different paths through the Brillouin zone, so they are
drawn side by side.

In [ ]:
import matplotlib.pyplot as plt
from pymatgen.phonon.bandstructure import PhononBandStructureSymmLine
from pymatgen.phonon.dos import PhononDos


def plot_bands(
    ax: plt.Axes, bs: PhononBandStructureSymmLine, label: str | None = None, **style
) -> None:
    """Plot a phonon band structure, one segment of the path at a time."""
    distance = np.array(bs.distance)
    positions, names = [], []
    for idx, branch in enumerate(bs.branches):
        part = slice(branch["start_index"], branch["end_index"] + 1)
        lines = ax.plot(distance[part], bs.bands[:, part].T, **style)
        if idx == 0 and label:
            lines[0].set_label(label)
        start, end = branch["name"].split("-")
        if positions and np.isclose(positions[-1], distance[part.start]):
            if names[-1] != start:
                names[-1] += "|" + start
        else:
            positions.append(distance[part.start])
            names.append(start)
        positions.append(distance[branch["end_index"]])
        names.append(end)
    names = [name.replace("\\Gamma", "Γ").replace("GAMMA", "Γ") for name in names]
    ax.set_xticks(positions, names)
    ax.set_xlim(distance[0], distance[-1])
    ax.axhline(0, color="gray", lw=0.5)


def plot_dos(ax: plt.Axes, dos: PhononDos, label: str | None = None, **style) -> None:
    """Plot a phonon DOS sideways, to sit next to a band structure."""
    ax.plot(dos.densities, dos.frequencies, label=label, **style)
    ax.set_xlabel("DOS")


with MPRester() as mpr:
    bs_0k = mpr.materials.phonon.get_bandstructure_from_material_id(
        "mp-1200211", phonon_method="pheasy"
    ).to_pmg
    dos_0k = mpr.materials.phonon.get_dos_from_material_id(
        "mp-1200211", phonon_method="pheasy"
    ).to_pmg

fig, axes = plt.subplots(
    1, 3, figsize=(13, 4), sharey=True, gridspec_kw={"width_ratios": [3, 3, 1]}
)
plot_bands(axes[0], bs_0k, color="gray", lw=0.8)
plot_bands(axes[1], doc.phonon_bandstructure, color="C0", lw=0.8)
plot_dos(axes[2], dos_0k, label="0 K", color="gray")
plot_dos(axes[2], doc.phonon_dos, label="300 K", color="C0")
axes[0].set_title("0 K, PBEsol")
axes[1].set_title("300 K, MACE-OMAT-0-medium")
axes[0].set_ylabel("Frequency (THz)")
axes[2].legend()
plt.show()

At 0 K, Ca3Ir4Sn13 has imaginary modes down to about -0.75 THz. At 300 K they
are gone. The motion of the atoms at 300 K stabilizes them.

To check the force field, we compare with effective harmonic force constants
fitted to ab initio MD at 300 K. These were computed with PBEsol in the same
cell and supercell. Their highest frequency is 6.03 THz, against 5.97 THz from
MACE-OMAT-0-medium. Over the whole band path the frequencies differ by 0.07 THz
on average.

## The other temperatures, potentials and materials

The same workflow runs at five temperatures, with all three potentials and for
four more materials. AlAsPt5, Zn3P2 and Er5Tl3 also have imaginary modes at 0 K.
Bi4S3N2 is polar. The function below builds and runs one workflow in its own
folder. Section 2 uses it as well.

The loop runs 75 workflows, so it is off by default. Set `RUN_ALL = True` to
run it. Each of our runs took 6 to 36 minutes on one A100 GPU.

In [ ]:
from atomate2.common.schemas.finite_temperature_phonons import (
    FiniteTemperaturePhononDoc,
)

# mp-id and the diagonal of the supercell matrix of our AIMD reference
MATERIALS = {
    "Ca3Ir4Sn13": ("mp-1200211", (2, 2, 2)),
    "AlAsPt5": ("mp-1025306", (4, 4, 2)),
    "Zn3P2": ("mp-2071", (2, 2, 2)),
    "Bi4S3N2": ("mp-1245549", (2, 2, 2)),
    "Er5Tl3": ("mp-1105965", (2, 2, 2)),
    "KNaICl": ("mp-1002081", (3, 3, 2)),
}
TEMPERATURES = (100, 300, 500, 700, 900)


def run_ft(
    structure: Structure,
    supercell: tuple[int, int, int],
    model: str,
    root_dir: str,
    temperature: float,
    md_time: float = 8.0,
    n_snapshots: int = 50,
) -> FiniteTemperaturePhononDoc:
    """Run the finite-temperature phonon workflow and return its output."""
    maker = ForceFieldFiniteTemperaturePhononMaker.from_force_field_name(
        "MACE-MP-0",
        calculator_kwargs=calc_kwargs(model),
        relax_initial_structure=False,
        temperature=temperature,
        md_time=md_time,
        n_snapshots=n_snapshots,
    )
    flow = maker.make(structure, supercell_matrix=np.diag(supercell).tolist())
    os.makedirs(root_dir, exist_ok=True)
    responses = run_locally(
        flow,
        store=JobStore(MemoryStore(), additional_stores={"data": MemoryStore()}),
        create_folders=True,
        root_dir=root_dir,
        ensure_success=True,
    )
    return responses[flow.output.uuid][1].output


RUN_ALL = False
docs = {}
if RUN_ALL:
    for name in ("Ca3Ir4Sn13", "AlAsPt5", "Zn3P2", "Bi4S3N2", "Er5Tl3"):
        mp_id, supercell = MATERIALS[name]
        unit_cell = get_pheasy_structure(mp_id)
        for model in MODEL_FILES:
            for temperature in TEMPERATURES:
                docs[name, model, temperature] = run_ft(
                    unit_cell,
                    supercell,
                    model,
                    f"runs/{name}_{model}_{temperature}K",
                    temperature,
                )

The next cell plots the results of the loop. For each potential it shows the
band structure at 100, 500 and 900 K. Ca3Ir4Sn13 and Zn3P2 have 40 atoms in the
primitive cell, so their 120 branches are hard to follow. For these two the DOS
at all five temperatures is shown as well.

In [ ]:
def plot_temperatures(name: str, with_dos: bool) -> None:
    """Plot the band structures of one material at 100, 500 and 900 K."""
    ratios = [3, 1] * len(MODEL_FILES) if with_dos else [3] * len(MODEL_FILES)
    fig, axes = plt.subplots(
        1,
        len(ratios),
        figsize=(4.5 * len(MODEL_FILES), 4),
        sharey=True,
        gridspec_kw={"width_ratios": ratios},
    )
    colors = dict(
        zip(TEMPERATURES, plt.cm.plasma(np.linspace(0, 0.85, 5)), strict=True)
    )
    step = 2 if with_dos else 1
    for idx, model in enumerate(MODEL_FILES):
        ax = axes[step * idx]
        for temperature in (100, 500, 900):
            plot_bands(
                ax,
                docs[name, model, temperature].phonon_bandstructure,
                label=f"{temperature} K",
                color=colors[temperature],
                lw=0.6,
            )
        ax.set_title(model)
        if with_dos:
            for temperature in TEMPERATURES:
                plot_dos(
                    axes[step * idx + 1],
                    docs[name, model, temperature].phonon_dos,
                    label=f"{temperature} K",
                    color=colors[temperature],
                )
    axes[0].set_ylabel("Frequency (THz)")
    axes[-1].legend(fontsize=8)
    fig.suptitle(name)
    plt.show()


if RUN_ALL:
    for name in ("Ca3Ir4Sn13", "AlAsPt5", "Zn3P2", "Bi4S3N2", "Er5Tl3"):
        plot_temperatures(name, with_dos=name in ("Ca3Ir4Sn13", "Zn3P2"))

## Our results

The next cell holds the results of our runs. For each material it gives the
highest frequency on the band path at each temperature. The dashed line is the
highest frequency of the AIMD reference at 300 K.

In [ ]:
import pandas as pd

# highest frequency on the band path in THz, at 100, 300, 500, 700 and 900 K
MAX_FREQUENCY = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": (5.98, 5.97, 5.93, 5.90, 5.83),
        "MATPES-PBE-0": (5.80, 5.75, 5.67, 5.63, 5.65),
        "MATPES-r2SCAN-0": (5.95, 5.98, 5.92, 5.88, 5.78),
    },
    "AlAsPt5": {
        "OMAT-0-medium": (7.75, 7.73, 7.81, 7.76, 7.71),
        "MATPES-PBE-0": (7.65, 7.57, 7.48, 7.59, 7.47),
        "MATPES-r2SCAN-0": (7.66, 7.62, 7.77, 7.53, 7.43),
    },
    "Zn3P2": {
        "OMAT-0-medium": (10.59, 10.36, 10.13, 9.67, 9.17),
        "MATPES-PBE-0": (10.33, 10.03, 9.86, 9.53, 8.73),
        "MATPES-r2SCAN-0": (11.19, 10.85, 10.66, 10.41, 9.97),
    },
    "Bi4S3N2": {
        "OMAT-0-medium": (17.15, 16.88, 16.05, 16.26, 15.47),
        "MATPES-PBE-0": (15.97, 15.64, 15.40, 14.75, 14.31),
        "MATPES-r2SCAN-0": (17.62, 16.94, 17.23, 16.87, 15.54),
    },
    "Er5Tl3": {
        "OMAT-0-medium": (4.23, 4.22, 4.19, 4.22, 4.23),
        "MATPES-PBE-0": (4.11, 4.07, 4.01, 3.93, 3.99),
        "MATPES-r2SCAN-0": (4.18, 4.09, 4.08, 3.99, 3.96),
    },
}
# AIMD reference at 300 K: highest frequency in THz, and the mean absolute
# difference of each potential from it over all branches of the band path
AIMD_MAX_300K = {
    "Ca3Ir4Sn13": 6.03,
    "AlAsPt5": 7.79,
    "Zn3P2": 10.09,
    "Bi4S3N2": 17.03,
    "Er5Tl3": 3.97,
}
MAE_300K = {
    "Ca3Ir4Sn13": {
        "OMAT-0-medium": 0.07,
        "MATPES-PBE-0": 0.17,
        "MATPES-r2SCAN-0": 0.10,
    },
    "AlAsPt5": {"OMAT-0-medium": 0.11, "MATPES-PBE-0": 0.21, "MATPES-r2SCAN-0": 0.18},
    "Zn3P2": {"OMAT-0-medium": 0.19, "MATPES-PBE-0": 0.09, "MATPES-r2SCAN-0": 0.42},
    "Bi4S3N2": {"OMAT-0-medium": 0.16, "MATPES-PBE-0": 0.43, "MATPES-r2SCAN-0": 0.17},
    "Er5Tl3": {"OMAT-0-medium": 0.17, "MATPES-PBE-0": 0.16, "MATPES-r2SCAN-0": 0.12},
}
# runs with imaginary modes on the band path, and runs whose trajectory check
# was not "stable"
IMAGINARY = {
    ("Bi4S3N2", "OMAT-0-medium"): (100, 300, 700),
    ("Bi4S3N2", "MATPES-PBE-0"): (300, 700, 900),
    ("Bi4S3N2", "MATPES-r2SCAN-0"): (100, 300, 500, 700, 900),
    ("Zn3P2", "MATPES-PBE-0"): (900,),
}
NOT_STABLE = {
    ("Ca3Ir4Sn13", "OMAT-0-medium", 100): "shifted_or_diffusing",
    ("Ca3Ir4Sn13", "MATPES-r2SCAN-0", 100): "shifted_or_diffusing",
    ("Zn3P2", "OMAT-0-medium", 700): "melted",
    ("Zn3P2", "OMAT-0-medium", 900): "melted",
    ("Zn3P2", "MATPES-PBE-0", 700): "melted",
    ("Zn3P2", "MATPES-PBE-0", 900): "melted",
    ("Zn3P2", "MATPES-r2SCAN-0", 700): "melted",
    ("Zn3P2", "MATPES-r2SCAN-0", 900): "melted",
    ("Bi4S3N2", "OMAT-0-medium", 700): "melted",
    ("Bi4S3N2", "OMAT-0-medium", 900): "melted",
    ("Bi4S3N2", "MATPES-PBE-0", 900): "melted",
    ("Bi4S3N2", "MATPES-r2SCAN-0", 900): "melted",
}

fig, axes = plt.subplots(1, 5, figsize=(16, 3.4))
for ax, (name, by_model) in zip(axes, MAX_FREQUENCY.items(), strict=True):
    for model, values in by_model.items():
        ax.plot(TEMPERATURES, values, "o-", label=model)
    ax.axhline(AIMD_MAX_300K[name], color="black", ls="--", lw=0.8)
    ax.set_title(name)
    ax.set_xlabel("Temperature (K)")
axes[0].set_ylabel("Highest frequency (THz)")
axes[0].legend(fontsize=8)
fig.tight_layout()
plt.show()

pd.DataFrame(MAE_300K).T.assign(**{"AIMD highest frequency": pd.Series(AIMD_MAX_300K)})

The table gives the mean absolute difference from the AIMD reference at 300 K
in THz, over all branches and q-points of the band path.

- No potential is closest to AIMD for all five materials. MACE-OMAT-0-medium is
  closest for Ca3Ir4Sn13, AlAsPt5 and Bi4S3N2, within 0.07 to 0.16 THz.
  MACE-MATPES-r2SCAN-0 is closest for Er5Tl3 and MACE-MATPES-PBE-0 for Zn3P2.
- MACE-MATPES-PBE-0 gives the lowest frequencies for most materials. For Bi4S3N2 its highest branches lie about
  1.4 THz below AIMD.
- Most frequencies soften with temperature. The highest frequency of Bi4S3N2
  drops by 1.7 THz from 100 K to 900 K with MACE-OMAT-0-medium. Er5Tl3 barely
  changes.
- The imaginary modes at 0 K are gone at every temperature for Ca3Ir4Sn13,
  AlAsPt5, Zn3P2 and Er5Tl3. The one exception is Zn3P2 with MACE-MATPES-PBE-0
  at 900 K, a run that the trajectory check reports as melted. Bi4S3N2 keeps imaginary modes down to -0.7 THz in
  11 of 15 runs. Its AIMD reference has a small one as well, at -0.12 THz.
  Bi4S3N2 is polar, so these runs and the AIMD reference are compared without
  the non-analytical correction. The force fields give no Born charges.

The trajectory check did not report "stable" for 12 runs:

- Zn3P2 and Bi4S3N2 are reported as melted at 700 K or 900 K. The Lindemann
  ratio is above 0.12 there. Zn3P2 melts well above 900 K. The limit of 0.12 is
  a rule of thumb, and we did not check whether these trajectories are liquid.
  Treat these runs with care.
- Ca3Ir4Sn13 at 100 K is reported as shifted or diffusing with two of the
  potentials. With MACE-OMAT-0-medium the mean positions of the atoms moved
  0.13 Å away from the starting structure. This is a little more than 1.5
  times their vibration of 0.11 Å. At 300 K the shift is only 0.05 Å, so it
  does not come from the small difference between the PBEsol and the force
  field positions. In experiment, Ca3Ir4Sn13 has a structural transition near
  40 K, with soft phonons above it (arXiv:1503.08102). The shift at 100 K may
  be the start of this distortion. We did not check this further.

## 2. MD length and number of snapshots for KNaICl

Two settings control how well the MD samples the temperature:

- `md_time` sets the length of the MD in ps. The default is 8 ps, of which the
  first 1 ps is left out.
- `n_snapshots` sets the number of snapshots used in the fit. The default is 50.

We tested 4, 8 and 16 ps with 25, 50 and 100 snapshots at 300 K for KNaICl
(mp-1002081). It has 4 atoms in the primitive cell and 72 in the 3x3x2
supercell. KNaICl is polar, so it is compared with the AIMD reference without
the non-analytical correction. The highest frequency of that reference is
5.51 THz.

The grid runs 27 workflows, so it is off by default.

In [ ]:
RUN_GRID = False
if RUN_GRID:
    mp_id, supercell = MATERIALS["KNaICl"]
    knaicl = get_pheasy_structure(mp_id)
    for model in MODEL_FILES:
        for md_time in (4, 8, 16):
            for n_snapshots in (25, 50, 100):
                docs["KNaICl", model, md_time, n_snapshots] = run_ft(
                    knaicl,
                    supercell,
                    model,
                    f"runs/KNaICl_{model}_{md_time}ps_{n_snapshots}",
                    300,
                    md_time,
                    n_snapshots,
                )

In [ ]:
MD_TIMES = (4, 8, 16)
N_SNAPSHOTS = (25, 50, 100)
# by potential, rows 4, 8 and 16 ps, columns 25, 50 and 100 snapshots
MAE_GRID = {  # mean absolute difference from AIMD at 300 K in THz
    "OMAT-0-medium": (
        (0.146, 0.149, 0.150),
        (0.159, 0.145, 0.144),
        (0.156, 0.155, 0.149),
    ),
    "MATPES-PBE-0": (
        (0.138, 0.121, 0.121),
        (0.124, 0.129, 0.127),
        (0.127, 0.135, 0.117),
    ),
    "MATPES-r2SCAN-0": (
        (0.137, 0.123, 0.123),
        (0.138, 0.131, 0.128),
        (0.147, 0.128, 0.128),
    ),
}
MAX_GRID = {  # highest frequency on the band path in THz
    "OMAT-0-medium": ((5.63, 5.65, 5.63), (5.63, 5.73, 5.70), (5.32, 5.43, 5.44)),
    "MATPES-PBE-0": ((5.03, 5.23, 5.26), (5.25, 5.22, 5.33), (5.12, 5.13, 5.27)),
    "MATPES-r2SCAN-0": ((5.23, 5.40, 5.40), (5.39, 5.66, 5.56), (5.27, 5.60, 5.54)),
}
IMAGINARY_GRID = {  # True if the band path has imaginary modes
    "OMAT-0-medium": ((False, True, True), (False, False, False), (True, True, False)),
    "MATPES-PBE-0": ((True, False, False), (True, True, True), (False, True, False)),
    "MATPES-r2SCAN-0": (
        (False, False, False),
        (True, True, False),
        (True, False, False),
    ),
}

fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
for ax, model in zip(axes, MODEL_FILES, strict=True):
    grid = np.array(MAE_GRID[model])
    image = ax.imshow(grid, cmap="viridis", vmin=0.11, vmax=0.16)
    for i in range(3):
        for j in range(3):
            mark = " *" if IMAGINARY_GRID[model][i][j] else ""
            ax.text(
                j, i, f"{grid[i, j]:.3f}{mark}", ha="center", va="center", color="w"
            )
    ax.set_xticks(range(3), N_SNAPSHOTS)
    ax.set_yticks(range(3), [f"{t} ps" for t in MD_TIMES])
    ax.set_xlabel("Number of snapshots")
    ax.set_title(model)
fig.colorbar(image, ax=axes, label="Mean absolute difference from AIMD (THz)")
plt.show()

pd.concat(
    {
        model: pd.DataFrame(
            MAX_GRID[model], index=[f"{t} ps" for t in MD_TIMES], columns=N_SNAPSHOTS
        )
        for model in MODEL_FILES
    },
    axis=1,
)

In the plot, a star marks a run with imaginary modes on the band path. The
table gives the highest frequency in THz.

- The mean absolute difference from AIMD lies between 0.12 and 0.16 THz in every
  run. It shows no trend with the MD length or the number of snapshots. Each
  run is one MD trajectory, so these differences are the noise of a single run.
- The highest frequency changes by up to 0.4 THz between runs, also without a
  trend.
- 12 of the 27 runs have imaginary modes on the band path, down to -0.9 THz.
  They sit at q-points that are not commensurate with the supercell, such as
  (1/2, 1/2, 0). There the frequencies are interpolated from the fitted force
  constants. The AIMD reference has no imaginary modes in the same supercell. So
  they come from the noise of the fit. They appear in 5 of 9 runs with 25
  snapshots, 5 of 9 with 50 and 2 of 9 with 100.
- The MD takes most of the time. With MACE-OMAT-0-medium a run took 3.8, 6.0
  and 10.6 minutes for 4, 8 and 16 ps. Going from 50 to 100 snapshots added a
  few seconds of force calculations and fitting.

We also ran Bi4S3N2 and Zn3P2 at 300 K with 100 snapshots. The mean absolute
difference from AIMD changed by 0.03 THz or less. The fit of Zn3P2 did not take
longer. Bi4S3N2 kept its imaginary modes with all three potentials, so these are
unlikely to be noise of the fit.

With a force field the force calculations cost little next to the MD, so 100
snapshots are worth using. The default stays at 50, because with VASP each
snapshot is a DFT calculation of the whole supercell. For the MD length, 4 ps
already gave the same agreement with AIMD for KNaICl. We kept the default of
8 ps for the other materials.

## Known limitations

The force constants are those of the starting volume. Thermal expansion is not
included, so the result is least reliable at high temperature.

A machine-learned potential gives no Born charges, so the non-analytical
correction is not applied here. With VASP, the `born_maker` computes them.

The trajectory check uses fixed limits. A Lindemann ratio above 0.12 is reported
as melting. A displacement from the reference of more than 1.5 times the
vibration around the mean positions is reported as a shift. These are rules of thumb. A warning is raised and
the fit is still done.

Every result above comes from one MD trajectory. Section 2 shows that two runs
with different settings can differ by about 0.04 THz on average and by up to
0.4 THz in the highest frequency.